# Q.ANT's NPU: Pixel-Art Generation with Diffusion

## Denoising Diffusion

A **diffusion model** learns to generate images by reversing a noising process:
- **Forward process**: starting from a training image $x_0$, Gaussian noise is added step by step until, after $T$ steps, only noise is left. Thanks to a closed form, any noisy step $x_t$ can be sampled directly:
$$x_t = \sqrt{\bar\alpha_t}\, x_0 + \sqrt{1 - \bar\alpha_t}\, \epsilon, \qquad \epsilon \sim \mathcal{N}(0, I)$$
- **Reverse process**: a neural network learns to denoise. Starting from pure noise, it is applied $T$ times to generate a new image.

Here the network gets a normal photo as an additional input and generates a pixel-art version of it (**image-conditional** diffusion).

We keep everything simple: the denoising network is a small multi-layer perceptron (MLP) built from `QLinear` layers. They are trained with standard PyTorch on the CPU, and in evaluation mode the matrix multiplications run on the NPU. Since generating an image calls the network $T$ times, the sampling loop is dominated by exactly these matrix multiplications.

# Let's start coding

#### Import required libraries and modules

In [ ]:
import math

import matplotlib.pyplot as plt
import numpy as np
import pixel_art_data  # Have a short look at this file to see how the dataset is built
import qant_layers  # Have a short look at this file to see how the layers are implemented
import torch

torch.manual_seed(10)
rng = np.random.default_rng(seed=20)

#### Pixel-Art Fruits Dataset

We build the training data from a few classes of the [Fruits-360](https://github.com/fruits-360/fruits-360-100x100) dataset by Mihai Oltean, licensed under [CC BY-SA 4.0](https://creativecommons.org/licenses/by-sa/4.0/). Only the selected classes are downloaded at runtime; the dataset is not redistributed with this example.

Each photo is turned into pixel art in three steps:
1. **Sharpen** the photo with an unsharp mask, so edges and highlights survive the downscaling.
2. **K-means palette quantization**: fit a palette of 16 colors with k-means on pixels of all images, then replace every pixel by its nearest palette color.
3. **Grid encoding**: split the image into a $16 \times 16$ grid and give each cell the most frequent palette color inside it. This keeps edges crisp, since no colors are blended.

Each training example is a pair: the photo downscaled to $32 \times 32$ pixels is the condition, and its pixel art is the target. The separate test split of Fruits-360 contains photos the model never sees during training; we use it to check that the model works on new photos.

In [ ]:
data_dir = pixel_art_data.download_fruits_360(pixel_art_data.DEFAULT_CLASSES)
pixel_art, photos, labels, palette = pixel_art_data.load_pixel_art_dataset(
    data_dir, pixel_art_data.DEFAULT_CLASSES, split="Training"
)
# The test split reuses the palette of the training split
test_pixel_art, test_photos, test_labels, _ = pixel_art_data.load_pixel_art_dataset(
    data_dir, pixel_art_data.DEFAULT_CLASSES, split="Test", palette=palette
)
class_names = [name.removesuffix(" 1") for name in pixel_art_data.DEFAULT_CLASSES]
print(f"{len(photos)} training pairs, {len(test_photos)} test pairs")
print(f"Photos: {photos.shape[1:]}, pixel art: {pixel_art.shape[1:]}")

#### Visualize the Training Data

A few training pairs, photo on top and pixel art below:

In [ ]:
def show_rows(rows: list[np.ndarray], row_names: list[str]) -> None:
    fig, axes = plt.subplots(len(rows), len(rows[0]), figsize=(len(rows[0]), len(rows)))
    for r, (row, name) in enumerate(zip(rows, row_names)):
        for ax, img in zip(axes[r], row):
            ax.imshow(img, interpolation="nearest")
            ax.axis("off")
        axes[r, 0].set_title(name, loc="left", fontsize=8)
    plt.tight_layout()
    plt.show()


# 2 random training pairs per class
show_idx = np.concatenate(
    [rng.choice(np.where(labels == c)[0], 2, replace=False) for c in range(len(class_names))]
)
show_rows([photos[show_idx], pixel_art[show_idx]], ["Photo", "Pixel art"])

#### Noise Schedule

We use $T = 100$ diffusion steps with a linear schedule for the noise variances $\beta_t$. With $\alpha_t = 1 - \beta_t$ and $\bar\alpha_t = \prod_{s \le t} \alpha_s$, the image is almost pure noise at the last step.

In [ ]:
T = 100
betas = torch.linspace(1e-3, 0.2, T)
alphas = 1 - betas
alpha_bars = torch.cumprod(alphas, dim=0)
print(f"Remaining signal at the last step: {alpha_bars[-1].sqrt():.1e}")

#### Denoising Network

The input of the MLP is the concatenation of
- the noisy image $x_t$, flattened to $16 \cdot 16 \cdot 3 = 768$ values in $[-1, 1]$,
- an embedding of the step $t$ (sine and cosine features, as in transformers),
- the photo, flattened to $32 \cdot 32 \cdot 3 = 3072$ values in $[-1, 1]$.

The network predicts the clean image $x_0$. For a small MLP this is easier to learn than predicting the noise $\epsilon$.

In [ ]:
def step_embedding(t: torch.Tensor, dim: int = 32) -> torch.Tensor:
    freqs = torch.exp(-math.log(1000) * torch.arange(dim // 2) / (dim // 2))
    args = t[:, None].float() * freqs[None, :]
    return torch.cat([torch.sin(args), torch.cos(args)], dim=1)


class Denoiser(torch.nn.Module):
    def __init__(self, n_pixels: int, n_photo: int, hidden: int = 512, t_dim: int = 32):
        super().__init__()
        self.t_dim = t_dim
        self.layers = torch.nn.ModuleList(
            [
                qant_layers.QLinear(n_pixels + t_dim + n_photo, hidden),
                qant_layers.QLinear(hidden, hidden),
                qant_layers.QLinear(hidden, n_pixels),
            ]
        )

    def forward(self, x: torch.Tensor, t: torch.Tensor, photo: torch.Tensor) -> torch.Tensor:
        h = torch.cat([x, step_embedding(t, self.t_dim), photo], dim=1)
        for layer in self.layers[:-1]:
            h = torch.relu(layer(h))
        return self.layers[-1](h)

#### Training

In every training step we take a random batch of training pairs, pick a random step $t$ for each image, noise the pixel art accordingly and train the network to recover the clean pixel art from the noisy one and the photo. Training runs on the CPU with standard PyTorch.

In [ ]:
def to_vectors(images: np.ndarray) -> torch.Tensor:
    """Flatten uint8 images to vectors with values in [-1, 1]."""
    return torch.from_numpy(images.reshape(len(images), -1)).float() / 127.5 - 1


x_train = to_vectors(pixel_art)
photo_train = to_vectors(photos)

model = Denoiser(n_pixels=x_train.shape[1], n_photo=photo_train.shape[1])
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

n_steps = 3000
batch_size = 128
train_losses = []
model.train()
for step in range(n_steps):
    idx = torch.randint(len(x_train), (batch_size,))
    x0, photo = x_train[idx], photo_train[idx]
    t = torch.randint(T, (batch_size,))
    noise = torch.randn_like(x0)
    alpha_bar = alpha_bars[t][:, None]
    xt = alpha_bar.sqrt() * x0 + (1 - alpha_bar).sqrt() * noise

    loss = torch.nn.functional.mse_loss(model(xt, t, photo), x0)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    train_losses.append(loss.item())

    if (step + 1) % 500 == 0:
        print(f"Step {step + 1}/{n_steps}, loss: {np.mean(train_losses[-500:]):.4f}")

In [ ]:
plt.plot(train_losses)
plt.xlabel("Training step")
plt.ylabel("MSE loss")
plt.yscale("log")
plt.show()

#### Sampling on the NPU

Switching the model to evaluation mode makes every `QLinear` layer run its matrix multiplication on the NPU. We generate pixel art for photos of the test split, which the model has never seen.

Starting from pure noise, each reverse step predicts the clean image $\hat x_0$ and moves to a slightly less noisy image $x_{t-1}$ by sampling from
$$\mathcal{N}\left(\frac{\sqrt{\bar\alpha_{t-1}}\,\beta_t\, \hat x_0 + \sqrt{\alpha_t}\,(1 - \bar\alpha_{t-1})\, x_t}{1 - \bar\alpha_t},\ \frac{1 - \bar\alpha_{t-1}}{1 - \bar\alpha_t}\,\beta_t\right).$$
At the last step, the prediction $\hat x_0$ is the generated image. Finally, every pixel is snapped to the nearest color of the palette.

In [ ]:
@torch.no_grad()
def sample(model: torch.nn.Module, photo: torch.Tensor) -> torch.Tensor:
    x = torch.randn(len(photo), x_train.shape[1])
    for t in reversed(range(T)):
        x0_hat = model(x, torch.full((len(photo),), t), photo).clamp(-1, 1)
        if t == 0:
            return x0_hat
        alpha_bar, alpha_bar_prev = alpha_bars[t], alpha_bars[t - 1]
        mean = (
            alpha_bar_prev.sqrt() * betas[t] * x0_hat
            + alphas[t].sqrt() * (1 - alpha_bar_prev) * x
        ) / (1 - alpha_bar)
        variance = betas[t] * (1 - alpha_bar_prev) / (1 - alpha_bar)
        x = mean + variance.sqrt() * torch.randn_like(x)


# 8 random test photos per class
test_idx = np.concatenate(
    [rng.choice(np.where(test_labels == c)[0], 8, replace=False) for c in range(len(class_names))]
)

model.eval()  # QLinear layers now run on the NPU
generated = sample(model, to_vectors(test_photos[test_idx]))

# Map from [-1, 1] back to RGB and snap to the palette
generated = ((generated + 1) * 127.5).numpy().reshape(-1, 16, 16, 3)
samples = pixel_art_data.quantize_to_palette(generated, palette)

# Fraction of pixels with the same palette color as the reference pixel art
pixel_accuracy = np.mean(np.all(samples == test_pixel_art[test_idx], axis=-1))
print(f"Pixel accuracy on test photos: {pixel_accuracy:.1%}")

In [ ]:
show = np.arange(0, len(test_idx), 4)  # 2 test photos per class
show_rows(
    [test_photos[test_idx][show], samples[show], test_pixel_art[test_idx][show]],
    ["Test photo", "Generated on NPU", "Reference pixel art"],
)